# TruncatedSVD LGBM Experiment

This notebook shows how to use `TruncatedSVD` as a dimensionality reduction method before training `LightGBM`.

`TruncatedSVD` is especially useful for wide sparse matrices because, unlike `PCA`, it does not center the matrix before decomposition. That means it can work naturally with sparse or mostly-zero feature sets.

## Learning Goals

After this notebook, you should understand:

- why dimensionality reduction can help with very wide anonymous feature sets;
- how `TruncatedSVD` differs from `PCA`;
- how to fit preprocessing and dimensionality reduction inside CV folds to avoid leakage;
- how to compare several values of `n_components` using CV RMSLE;
- how to evaluate the selected setup on a held-out test split.

In [1]:
# Add the project root to Python path so notebook imports can see the local src package.
import sys

sys.path.append("../")

# NumPy is used for log-transforming the target, inverse transforms, and CV aggregation.
import numpy as np
# Pandas is used for tabular data manipulation and readable result tables.
import pandas as pd

In [2]:
# SciPy sparse matrices let TruncatedSVD work efficiently with mostly-zero wide data.
from scipy import sparse
# TruncatedSVD performs dimensionality reduction without centering the matrix.
from sklearn.decomposition import TruncatedSVD
# Metrics are calculated on the original target scale after inverse log transform.
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
# train_test_split creates the final holdout; KFold drives CV on the training split.
from sklearn.model_selection import KFold, train_test_split
# with_mean=False preserves sparse structure by avoiding feature centering.
from sklearn.preprocessing import StandardScaler

# Project helpers keep loading, formatting, preprocessing, and model construction consistent.
from src.loader import Loader
from src.metrics import format_metric_value
from src.modeling import build_lgbm_regressor
from src.preprocessing import FeaturePreprocessor

## Configuration

The model parameters are fixed on purpose. This notebook measures the effect of SVD component count, not another hyperparameter search.

`SPARSITY_THRESHOLD=0.9875` follows the earlier sparsity experiment. The threshold is fitted inside each CV fold, so validation rows do not influence which columns are kept.

In [3]:
# Fixed seed makes train/test split, CV folds, and SVD initialization reproducible.
SEED = 42
# Keep one third of rows as a final sanity-check holdout set.
TEST_SIZE = 0.33
# Use 5-fold CV to choose the SVD component count on the training split only.
CV = 5

# Remove columns that are almost always zero before dimensionality reduction.
SPARSITY_THRESHOLD = 0.9875
# Candidate dimensionalities: each value means the model sees only this many SVD features.
SVD_COMPONENTS = [10, 25, 50, 100, 200]
# Human-readable label for the exact dimensionality-reduction pipeline being tested.
SVD_MODE = "svd_only_after_sparse_filter"

# Fixed LightGBM parameters: SVD_COMPONENTS is the variable being tested here.
LGBM_PARAMS = {
    # Number of boosting rounds.
    "n_estimators": 700,
    # Small learning rate for a smoother, more conservative fit.
    "learning_rate": 0.01,
    # Tree complexity controls.
    "num_leaves": 31,
    "max_depth": 6,
    "min_child_samples": 40,
    # Row and column sampling reduce variance and overfitting risk.
    "subsample": 0.75,
    "subsample_freq": 1,
    "colsample_bytree": 0.75,
    # Regularization terms penalize overly complex trees.
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "min_split_gain": 0.05,
}

## Load Data

The project already has a processed dataset where `target` is the regression target and all other columns are numeric model features.

In [4]:
# Load the already processed modeling table produced by earlier notebooks.
loader = Loader()
df = loader.load("../data/processed_data.csv")
# Show rows and columns as a quick sanity check.
df.shape

(4459, 4732)

In [5]:
# All columns except target are input features.
X = df.drop(columns="target")
# Keep the raw target for final metric calculation on the original scale.
y = df["target"]
# Train LightGBM on log1p(target); RMSE in this space corresponds to RMSLE.
y_log = np.log1p(y)

# Confirm the feature matrix and target vector dimensions.
(X.shape, y.shape)

((4459, 4731), (4459,))

## Train/Test Split

The test split is held out until the end. Component selection is done with cross-validation on the training split only.

In [6]:
# Split both raw target and log target so each is available later.
X_train_raw, X_test_raw, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

# Shuffle folds so each fold has a more representative mix of rows.
cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

## TruncatedSVD vs PCA

`PCA` centers every feature before decomposition. Centering turns sparse matrices into dense matrices, which can be expensive for wide mostly-zero data.

`TruncatedSVD` decomposes the matrix without centering. Because of that, it is commonly used for sparse text matrices and other high-dimensional sparse feature sets.

In this notebook the steps are:

1. fit sparse-column filtering on the training part only;
2. scale features with `StandardScaler(with_mean=False)` so sparsity is preserved;
3. convert the matrix to CSR sparse format;
4. fit `TruncatedSVD` on the training part only;
5. train LightGBM on SVD components only.

## SVD Helpers

The helper below receives a fit matrix and a transform matrix. It fits all preprocessing objects only on `X_fit`, then applies the fitted objects to `X_transform`.

This pattern is important in CV: each validation fold must behave like unseen data.

In [7]:
def build_svd_features(
    X_fit: pd.DataFrame,
    X_transform: pd.DataFrame,
    n_components: int,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    # Fit sparse-column filtering only on X_fit to avoid validation/test leakage.
    preprocessor = FeaturePreprocessor(
        zero_share_threshold=SPARSITY_THRESHOLD,
        add_rowwise=False,
    )
    X_fit_filtered = preprocessor.fit_transform(X_fit)
    # Apply the fitted column selection to the second matrix.
    X_transform_filtered = preprocessor.transform(X_transform)

    # Scale features without centering; centering would destroy sparsity.
    scaler = StandardScaler(with_mean=False)
    X_fit_scaled = scaler.fit_transform(X_fit_filtered)
    # Use the fold/train-fitted scaler for validation or test rows.
    X_transform_scaled = scaler.transform(X_transform_filtered)

    # Convert to CSR format, the common sparse matrix format expected by SVD workflows.
    X_fit_sparse = sparse.csr_matrix(X_fit_scaled)
    X_transform_sparse = sparse.csr_matrix(X_transform_scaled)

    # Fit SVD basis only on X_fit, then project both matrices into that basis.
    svd = TruncatedSVD(n_components=n_components, random_state=SEED)
    X_fit_svd = svd.fit_transform(X_fit_sparse)
    X_transform_svd = svd.transform(X_transform_sparse)

    # Give components stable names so downstream tables are readable.
    svd_columns = [f"svd_{idx + 1:03d}" for idx in range(n_components)]
    # Convert arrays back to DataFrames and preserve original indexes for alignment.
    X_fit_svd = pd.DataFrame(X_fit_svd, columns=svd_columns, index=X_fit.index)
    X_transform_svd = pd.DataFrame(X_transform_svd, columns=svd_columns, index=X_transform.index)

    # Keep preprocessing diagnostics for later reporting and interpretation.
    diagnostics = {
        "base_features_kept": len(preprocessor.columns_to_keep_),
        "base_features_removed": len(preprocessor.removed_sparse_columns_),
        "explained_variance": float(svd.explained_variance_ratio_.sum()),
    }

    # Return both transformed matrices plus diagnostics from the fitted pipeline.
    return X_fit_svd, X_transform_svd, diagnostics

In [8]:
def evaluate_cv(n_components: int) -> dict:
    # Store one validation score and diagnostic values per fold.
    fold_scores = []
    explained_variance = []
    base_features_kept = []

    # CV is run only on the training split; the test split remains untouched until the end.
    for fold_train_idx, fold_valid_idx in cv.split(X_train_raw, y_train_log):
        # Select fold-specific train and validation rows.
        X_fold_train_raw = X_train_raw.iloc[fold_train_idx]
        X_fold_valid_raw = X_train_raw.iloc[fold_valid_idx]
        y_fold_train_log = y_train_log.iloc[fold_train_idx]
        y_fold_valid_log = y_train_log.iloc[fold_valid_idx]

        # Fit sparse filtering, scaling, and SVD on fold train; transform fold validation.
        X_fold_train, X_fold_valid, diagnostics = build_svd_features(
            X_fold_train_raw,
            X_fold_valid_raw,
            n_components=n_components,
        )

        # Train LightGBM on SVD components only.
        model = build_lgbm_regressor(LGBM_PARAMS)
        model.fit(X_fold_train, y_fold_train_log)
        y_fold_pred_log = model.predict(X_fold_valid)

        # Log-space RMSE is the CV optimization metric for the log1p target setup.
        fold_scores.append(root_mean_squared_error(y_fold_valid_log, y_fold_pred_log))
        # Store fold diagnostics to understand how much information SVD retained.
        explained_variance.append(diagnostics["explained_variance"])
        base_features_kept.append(diagnostics["base_features_kept"])

    # Return aggregate CV diagnostics for this component count.
    return {
        "n_components": n_components,
        "cv_rmsle_mean": float(np.mean(fold_scores)),
        "cv_rmsle_std": float(np.std(fold_scores)),
        "explained_variance_mean": float(np.mean(explained_variance)),
        "base_features_kept_mean": float(np.mean(base_features_kept)),
    }

## CV Comparison

Lower `cv_rmsle_mean` is better. `explained_variance_mean` shows how much input variance is captured by the selected SVD components.

A larger number of components usually captures more variance, but it can also make the model slower and can overfit.

In [9]:
# Collect CV results for every candidate number of SVD components.
results = []

for n_components in SVD_COMPONENTS:
    results.append(evaluate_cv(n_components=n_components))

# Sort by mean CV RMSLE; the first row is the best candidate.
results_df = pd.DataFrame(results).sort_values("cv_rmsle_mean")
results_df

,n_components,cv_rmsle_mean,cv_rmsle_std,explained_variance_mean,base_features_kept_mean
2,50,1.525754,0.054069,0.357528,2498.2
3,100,1.528413,0.052009,0.482327,2498.2
1,25,1.535061,0.048001,0.261099,2498.2
4,200,1.536022,0.045113,0.633768,2498.2
0,10,1.551747,0.052928,0.182656,2498.2


In [10]:
# Display the same table with compact metric formatting for easier notebook reading.
results_df.style.format(
    {
        "cv_rmsle_mean": "{:.4f}",
        "cv_rmsle_std": "{:.4f}",
        "explained_variance_mean": "{:.4f}",
        "base_features_kept_mean": "{:,.0f}",
    }
).hide(axis="index")

n_components,cv_rmsle_mean,cv_rmsle_std,explained_variance_mean,base_features_kept_mean
50,1.5258,0.0541,0.3575,"2,498"
100,1.5284,0.0520,0.4823,"2,498"
25,1.5351,0.0480,0.2611,"2,498"
200,1.5360,0.0451,0.6338,"2,498"
10,1.5517,0.0529,0.1827,"2,498"


## Final Test Evaluation

After choosing `best_n_components` from CV, fit the full SVD pipeline on the training split and evaluate once on the held-out test split.

In [11]:
# Choose the best component count based on CV, not on the test split.
best_row = results_df.iloc[0]
best_n_components = int(best_row["n_components"])

# Fit the full sparse-filtering, scaling, and SVD pipeline on the training split.
X_train_final, X_test_final, final_diagnostics = build_svd_features(
    X_train_raw,
    X_test_raw,
    n_components=best_n_components,
)

# Train the final model using only the selected SVD components.
final_model = build_lgbm_regressor(LGBM_PARAMS)
final_model.fit(X_train_final, y_train_log)

# Predict in log space, then invert log1p with expm1 to return to the original target scale.
y_train_pred_log = final_model.predict(X_train_final)
y_train_pred = np.expm1(y_train_pred_log)
# Clipping prevents invalid negative predictions for RMSLE.
y_train_pred = np.clip(y_train_pred, 0, None)

# Repeat the same prediction conversion for the held-out test split.
y_test_pred_log = final_model.predict(X_test_final)
y_test_pred = np.expm1(y_test_pred_log)
y_test_pred = np.clip(y_test_pred, 0, None)

In [12]:
# RMSLE is the primary metric and matches the log-target training objective.
train_rmsle = root_mean_squared_log_error(y_train_raw, y_train_pred)
test_rmsle = root_mean_squared_log_error(y_test_raw, y_test_pred)
# RMSE and MAE are reported on the original target scale for business interpretability.
train_rmse = root_mean_squared_error(y_train_raw, y_train_pred)
test_rmse = root_mean_squared_error(y_test_raw, y_test_pred)
train_mae = mean_absolute_error(y_train_raw, y_train_pred)
test_mae = mean_absolute_error(y_test_raw, y_test_pred)
# R2 gives a familiar variance-explained view, but RMSLE remains the decision metric.
train_r2 = r2_score(y_train_raw, y_train_pred)
test_r2 = r2_score(y_test_raw, y_test_pred)

# Put all final diagnostics into one long table for easy display and report copying.
metrics = pd.DataFrame(
    {
        "metric": [
            "sparsity_threshold",
            "base_features_kept",
            "base_features_removed",
            "best_n_components",
            "explained_variance",
            "final_features",
            "cv_rmsle_mean",
            "cv_rmsle_std",
            "train_rmsle",
            "test_rmsle",
            "rmsle_gap_test_minus_train",
            "train_rmse",
            "test_rmse",
            "rmse_gap_test_minus_train",
            "train_mae",
            "test_mae",
            "train_r2",
            "test_r2",
        ],
        "value": [
            SPARSITY_THRESHOLD,
            final_diagnostics["base_features_kept"],
            final_diagnostics["base_features_removed"],
            best_n_components,
            final_diagnostics["explained_variance"],
            X_train_final.shape[1],
            best_row["cv_rmsle_mean"],
            best_row["cv_rmsle_std"],
            train_rmsle,
            test_rmsle,
            test_rmsle - train_rmsle,
            train_rmse,
            test_rmse,
            test_rmse - train_rmse,
            train_mae,
            test_mae,
            train_r2,
            test_r2,
        ],
    }
)

# Reuse the project formatter so large currency-like errors and small metrics are readable.
metrics.style.format({"value": format_metric_value}).hide(axis="index")

metric,value
sparsity_threshold,0.9875
base_features_kept,"2,482"
base_features_removed,"2,249"
best_n_components,50
explained_variance,0.3372
final_features,50
cv_rmsle_mean,1.5258
cv_rmsle_std,0.0541
train_rmsle,1.0628
test_rmsle,1.5066


## Summary

The summary dictionary makes it easy to copy the result into the final experiment report after running the notebook.

In [13]:
# Machine-readable summary of the experiment. Use this when updating final reports.
summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "model": "LGBMRegressor",
    "dimensionality_reduction": "TruncatedSVD",
    "svd_mode": SVD_MODE,
    "svd_components": SVD_COMPONENTS,
    "cv_folds": CV,
    "test_size": TEST_SIZE,
    "seed": SEED,
    "sparsity_threshold": SPARSITY_THRESHOLD,
    "model_params": LGBM_PARAMS,
    "best_n_components": best_n_components,
    "base_features_kept": final_diagnostics["base_features_kept"],
    "base_features_removed": final_diagnostics["base_features_removed"],
    "explained_variance": float(final_diagnostics["explained_variance"]),
    "final_features": X_train_final.shape[1],
    "cv_rmsle_mean": float(best_row["cv_rmsle_mean"]),
    "cv_rmsle_std": float(best_row["cv_rmsle_std"]),
    "train_rmsle": float(train_rmsle),
    "test_rmsle": float(test_rmsle),
    "rmsle_gap_test_minus_train": float(test_rmsle - train_rmsle),
    "train_rmse": float(train_rmse),
    "test_rmse": float(test_rmse),
    "rmse_gap_test_minus_train": float(test_rmse - train_rmse),
    "train_mae": float(train_mae),
    "test_mae": float(test_mae),
    "train_r2": float(train_r2),
    "test_r2": float(test_r2),
}

summary

{'target_transform': 'log1p',
 'primary_metric': 'rmsle',
 'model': 'LGBMRegressor',
 'dimensionality_reduction': 'TruncatedSVD',
 'svd_mode': 'svd_only_after_sparse_filter',
 'svd_components': [10, 25, 50, 100, 200],
 'cv_folds': 5,
 'test_size': 0.33,
 'seed': 42,
 'sparsity_threshold': 0.9875,
 'model_params': {'n_estimators': 700,
  'learning_rate': 0.01,
  'num_leaves': 31,
  'max_depth': 6,
  'min_child_samples': 40,
  'subsample': 0.75,
  'subsample_freq': 1,
  'colsample_bytree': 0.75,
  'reg_alpha': 0.1,
  'reg_lambda': 5.0,
  'min_split_gain': 0.05},
 'best_n_components': 50,
 'base_features_kept': 2482,
 'base_features_removed': 2249,
 'explained_variance': 0.337223330107245,
 'final_features': 50,
 'cv_rmsle_mean': 1.5257536221326862,
 'cv_rmsle_std': 0.054069312857841584,
 'train_rmsle': 1.062835637435614,
 'test_rmsle': 1.5066368500468152,
 'rmsle_gap_test_minus_train': 0.4438012126112012,
 'train_rmse': 6838856.165179571,
 'test_rmse': 7564524.934131713,
 'rmse_gap_test_

## How To Read The Result

- Compare `cv_rmsle_mean` against notebook `03` for the original-feature baseline and notebook `08` for the current tuned LightGBM setup.
- Compare this notebook against `10_pca_lgbm.ipynb` to see whether non-centered SVD is better than centered PCA for this dataset.
- `explained_variance` is useful for diagnostics, but the final decision should be based on CV RMSLE.
- A large positive `rmsle_gap_test_minus_train` means the model fits train much better than unseen test data.
- Treat `test_rmsle` as a sanity check, not as the optimization target.

# 11 TruncatedSVD LGBM Report

## Goal

The goal of this notebook is to test whether `TruncatedSVD` components can replace the sparse anonymous feature set for LightGBM regression.

## What Was Done

- Loaded `data/processed_data.csv`.
- Split the data into train and test parts with `test_size=0.33` and seed `42`.
- Used `log1p(target)` and optimized log-space RMSE, equivalent to RMSLE for this target transform.
- Fitted sparse filtering inside each CV fold with threshold `0.9875`.
- Scaled features with `StandardScaler(with_mean=False)` to preserve sparse structure.
- Converted the feature matrix to CSR sparse format.
- Fitted `TruncatedSVD` inside each CV fold to avoid leakage.
- Tested SVD component counts: `10`, `25`, `50`, `100`, `200`.
- Trained a fixed conservative `LGBMRegressor` on SVD components only.

## Main Results

| n_components | CV RMSLE mean | CV RMSLE std | CV explained variance mean | mean kept base features |
| ---: | ---: | ---: | ---: | ---: |
| 50 | 1.5258 | 0.0541 | 0.3575 | 2,498.2 |
| 100 | 1.5284 | 0.0520 | 0.4823 | 2,498.2 |
| 25 | 1.5351 | 0.0480 | 0.2611 | 2,498.2 |
| 200 | 1.5360 | 0.0451 | 0.6338 | 2,498.2 |
| 10 | 1.5517 | 0.0529 | 0.1827 | 2,498.2 |

The best CV candidate was `50` SVD components with mean CV RMSLE `1.5258`. In the final train/test fit, sparse filtering kept `2,482` original features and removed `2,249`; the selected `50` components explained `0.3372` of the train-split variance.

| metric | value |
| --- | ---: |
| Sparsity threshold | 0.9875 |
| Base features kept | 2,482 |
| Base features removed | 2,249 |
| Best SVD components | 50 |
| Explained variance | 0.3372 |
| Final features | 50 |
| CV RMSLE mean | 1.5258 |
| CV RMSLE std | 0.0541 |
| Train RMSLE | 1.0628 |
| Test RMSLE | 1.5066 |
| RMSLE gap test-train | 0.4438 |
| Train RMSE | 6,838,856.17 |
| Test RMSE | 7,564,524.93 |
| RMSE gap test-train | 725,668.77 |
| Train MAE | 3,550,488.41 |
| Test MAE | 4,266,967.21 |
| Train R2 | 0.3293 |
| Test R2 | 0.1034 |

## Conclusion

SVD-only should not be accepted as a replacement for the original-feature LightGBM setup. The best SVD CV RMSLE is `1.5258`, which is slightly better than the PCA-only result from notebook `10` (`1.5471`) but still materially worse than the tuned original-feature runs around `1.36`. The held-out test RMSLE is also weak at `1.5066`, and the train/test RMSLE gap of `0.4438` shows strong generalization degradation.

Increasing the number of SVD components preserved more input variance, up to `0.6338` at `200` components, but did not improve CV RMSLE. This suggests that the compressed components retain broad variance while losing sparse target-relevant signals. The better follow-up experiment is to add SVD components to the original filtered features, not to use SVD as a full feature replacement.